# 大模型算法工程师常见算法题

这个 notebook 聚焦大模型算法工程师面试里常见的“算法题/原理题/手写题”，尤其是 Transformer、MHA、KV Cache、位置编码、并行训练、采样与推理优化。

适用方向：
- 多模态/LLM 算法工程师
- 模型训练/推理优化工程师
- 视频理解、多模态 caption、VLM 相关岗位

建议准备方式：
1. 能口头讲清楚公式、shape、复杂度。
2. 能手写核心模块的简化版代码。
3. 能说明训练和推理阶段的工程 tradeoff。
4. 能把 MHA、RoPE、KV cache、FlashAttention 串成一个完整故事。

## 1. 一般会考什么

大模型算法工程师的面试题通常分成 5 类：

### A. Transformer 基础结构
- 为什么 self-attention 比 RNN 更适合长距离依赖
- Transformer block 的结构是什么
- Pre-LN 和 Post-LN 的差异
- residual connection 的作用
- FFN 为什么通常升维到 4 倍

### B. 注意力机制
- scaled dot-product attention 公式
- 为什么要除以 `sqrt(d_k)`
- MHA 和 MQA/GQA 的区别
- causal mask 是什么，为什么推理时必须要用
- attention 的时间复杂度和显存复杂度

### C. 位置编码
- 绝对位置编码 vs 相对位置编码
- RoPE 的原理，为什么适合外推
- ALiBi 的思想
- 长上下文时为什么位置编码会成为瓶颈

### D. 训练与推理工程
- KV cache 存的是什么，节省了什么
- FlashAttention 为什么更快
- 张量并行、流水并行、数据并行分别解决什么问题
- 混合精度训练为什么可行
- 推理阶段 prefill 和 decode 的差别

### E. 手写实现/代码题
- 手写 scaled dot-product attention
- 手写 multi-head attention
- 手写 causal mask
- 手写 top-k / top-p sampling
- 手写 RoPE 的 apply 逻辑
- 给定输入 shape，推导各层输出 shape

## 2. Self-Attention 是什么

面试里经常不会一上来就问 MHA，而是先问 self-attention 本身。

### 一句话定义
- self-attention 的核心是：序列中的每个 token 都可以直接看见其他 token，并根据相关性对它们加权聚合。

### 为什么它有效
- 能直接建模长距离依赖，不像 RNN 需要一步步传递。
- 适合并行计算。
- 对复杂关系建模更灵活。

### 面试时推荐回答
1. 输入序列先映射成 `Q/K/V`。
2. 用 `Q` 和 `K` 计算相似度，得到 attention score。
3. 对 score 做 softmax，得到权重。
4. 用权重对 `V` 加权求和。
5. 最终每个 token 得到一个融合全局上下文的新表示。

### 常见追问
- 为什么 self-attention 比 RNN 更适合长序列。
- 它的复杂度为什么高。
- 它为什么必须配合位置编码。

## 3. Scaled Dot-Product Attention

这是 attention 里最基础、也最容易被要求手写的一部分。

### 标准公式

`Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V`

### 每一项是什么意思
- `Q`：当前 token 想查询什么
- `K`：每个 token 能提供什么线索
- `V`：每个 token 真正携带的内容
- `QK^T`：相关性分数
- `softmax`：归一化成权重
- `乘 V`：把内容按权重聚合回来

### 为什么要除以 `sqrt(d_k)`
- 点积方差会随维度增大而增大。
- score 太大会让 softmax 饱和。
- softmax 一旦过尖，梯度会变差。
- 所以要做缩放。

### shape 推导
- `Q`: `(B, H, T_q, d)`
- `K`: `(B, H, T_k, d)`
- `V`: `(B, H, T_k, d_v)`
- `QK^T`: `(B, H, T_q, T_k)`
- 输出: `(B, H, T_q, d_v)`


In [1]:
import math
import torch


def scaled_dot_product_attention(q, k, v, mask=None):
    """
    q: (B, H, Tq, d)
    k: (B, H, Tk, d)
    v: (B, H, Tk, dv)
    mask: broadcastable to (B, H, Tq, Tk)
    """
    d = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = torch.softmax(scores, dim=-1)
    out = torch.matmul(attn, v)
    return out, attn


B, H, Tq, Tk, d = 2, 4, 8, 8, 16
q = torch.randn(B, H, Tq, d)
k = torch.randn(B, H, Tk, d)
v = torch.randn(B, H, Tk, d)
mask = torch.tril(torch.ones(Tq, Tk)).view(1, 1, Tq, Tk)

out, attn = scaled_dot_product_attention(q, k, v, mask)
print('out shape:', out.shape)
print('attn shape:', attn.shape)

out shape: torch.Size([2, 4, 8, 16])
attn shape: torch.Size([2, 4, 8, 8])


## 4. MHA 是最高频题之一

一个标准问题通常会这么问：

- 请你从输入 `X` 开始，推导 Multi-Head Attention 的完整计算流程。
- 每一步的张量 shape 是什么。
- 为什么要分多头。
- 为什么 attention score 要除以 `sqrt(d_k)`。
- 推理时怎么配合 KV cache。

### 标准公式

给定输入：
- `X`: `(B, T, D)`
- 头数 `H`
- 每个 head 的维度 `d = D / H`

线性映射：
- `Q = X W_Q`
- `K = X W_K`
- `V = X W_V`

reshape 后：
- `Q, K, V`: `(B, H, T, d)`

attention score：
- `S = Q K^T / sqrt(d)`
- `S`: `(B, H, T, T)`

加 mask 后 softmax：
- `A = softmax(S + mask)`

加权求和：
- `O = A V`
- `O`: `(B, H, T, d)`

拼接 heads：
- `O -> (B, T, D)`
- 再过输出投影 `W_O`

### 高频追问
- 为什么是多头而不是单头：不同子空间建模不同关系。
- 为什么除 `sqrt(d)`：防止点积随维度增大导致 softmax 饱和。
- 为什么复杂度高：attention matrix 是 `T x T`。
- 为什么 decode 时可以用 KV cache：历史 token 的 K/V 不变，不需要重复计算。

## 5. Transformer 推理复杂度

这类问题在大模型、语音、多模态岗位里都很常见。

### 训练时复杂度
- attention score 的核心复杂度是 `O(B * T^2 * D)`。
- 真正的大头通常来自 `T^2`。

### 推理时要分成 prefill 和 decode

#### prefill
- 一次性处理整段 prompt。
- 复杂度仍然接近 `O(T^2 * D)`。
- 瓶颈通常是 attention 计算本身。

#### decode
- 每次只生成一个 token。
- 如果有 KV cache，第 `t` 步只需要当前 `Q` 和历史 `K/V` 交互。
- 单步复杂度更像 `O(t * D)`。
- 生成整段输出时，累计下来大约是 `O(T^2 * D)`，但每步延迟大幅下降。

### 面试时推荐回答
1. 先说标准 Transformer 的瓶颈是 `T^2`。
2. 再说推理要区分 prefill 和 decode。
3. 然后补 KV cache 如何减少重复计算。
4. 最后补一句长上下文推理常常更受 KV cache 显存限制。

In [ ]:
def attention_flops_per_layer(batch_size, seq_len, d_model):
    # 粗略估算 attention 主项复杂度
    return batch_size * (seq_len ** 2) * d_model


for seq_len in [512, 2048, 8192]:
    print(seq_len, attention_flops_per_layer(batch_size=1, seq_len=seq_len, d_model=4096))

## 6. MHA 和 MLA 怎么理解

如果面试官问到 `MLA`，通常不是让你复现 DeepSeek 论文细节，而是看你有没有理解它在推理效率上的动机。

### MHA
- 每个 head 都有自己的 `Q/K/V`。
- 表达能力强。
- 但 KV cache 成本高。

### MLA
- 可以把它理解成：在保持较强表达能力的同时，尽量压缩或重参数化 KV 表示，降低推理时的 KV cache 开销。
- 它的动机和 MQA/GQA 有相通之处，都是在想办法降低长上下文推理成本。
- 面试里更重要的是讲清“为什么需要它”，而不是硬背论文细节。

### 更安全的回答方式
- MHA 的问题是推理时 KV cache 成本高。
- MLA 这类设计的核心目标，是在尽量保留多头表达能力的同时，降低 KV 表示的存储和访存成本。
- 所以它本质上是在做表达能力和推理效率的 tradeoff。

### 面试时别说太死的点
- 如果你没有系统看过 DeepSeek MLA 的公式推导，不要硬讲特别细的实现。
- 更稳的说法是从动机、推理成本、KV cache 角度去解释。

## 7. MHA / GQA / MQA / MLA 对比表

| 机制 | Q 头数 | K/V 头数 | 优点 | 缺点 | 适合怎么答 |
|---|---:|---:|---|---|---|
| MHA | 多 | 多 | 表达能力最强，最标准 | KV cache 成本高 | 先讲标准基线 |
| GQA | 多 | 少于 Q 头 | 在效果和 KV 成本之间折中 | 比 MHA 有一定表达损失 | 重点讲 tradeoff |
| MQA | 多 | 1 组共享 K/V | KV cache 最省，推理更快 | 表达能力可能掉更多 | 重点讲长上下文推理 |
| MLA | 多 | 不直接等同于标准多头 KV | 目标是尽量保留表达能力同时降低 KV 成本 | 实现更复杂，不适合硬背细节 | 从动机和推理效率讲 |

### 推荐口语版
- `MHA` 是标准方案，表达能力强，但推理时 KV cache 很贵。
- `GQA` 和 `MQA` 都是在降低 K/V 头数，本质是用一点表达能力换推理效率。
- `MLA` 这类设计的核心目标也是降低推理成本，但它通常不是简单把 K/V 头数减小，而是做更复杂的表示压缩或重参数化。
- 所以答题重点不是死记结构，而是说清楚：它们都在做表达能力和推理成本之间的 tradeoff。

## 8. Transformer 推理复杂度 + KV Cache + 显存占用总结

这部分适合当成一页总结来背。

### 1. 训练阶段
- attention 主复杂度：`O(B * T^2 * D)`
- attention 矩阵显存也是 `T^2` 级别
- 所以长序列训练首先卡在 attention 计算和显存

### 2. 推理阶段分成两段

#### prefill
- 一次性吃完整段 prompt
- 复杂度接近 `O(T^2 * D)`
- 更像训练时 attention 的模式

#### decode
- 每次只生成一个 token
- 有 KV cache 后，不需要重算历史 K/V
- 第 `t` 步只和历史长度 `t` 交互，单步更像 `O(t * D)`
- 所以 decode 的瓶颈往往更偏 memory / KV 读写，而不只是纯算力

### 3. KV cache 存了什么
- 每层、每个 token 的 `K` 和 `V`
- 不缓存 `Q`，因为 `Q` 只服务当前步

### 4. KV cache 为什么会成为长上下文瓶颈
- 它随 batch、seq len、layer 数、KV head 数、head dim 线性增长
- 所以公式可以记成：

`KV cache bytes ~= B * T * L * H_kv * d * 2 * dtype_bytes`

其中：
- `B`：batch size
- `T`：上下文长度
- `L`：层数
- `H_kv`：KV 头数
- `d`：head dim
- `2`：K 和 V 各一份

### 5. 为什么 GQA / MQA / MLA 有价值
- 因为它们都在想办法降低 `H_kv` 或等价降低 KV 表示成本
- 长上下文推理里，这通常比单纯优化 FLOPs 更关键

### 6. 面试时最安全的回答框架
1. 先说标准 Transformer 的 attention 训练复杂度是 `T^2`。
2. 再说推理要分 prefill 和 decode。
3. 然后补 KV cache 用显存换延迟。
4. 最后说长上下文瓶颈往往落到 KV cache，而 GQA / MQA / MLA 都是在优化这一点。

## 9. MHA 是最高频题之一

一个标准问题通常会这么问：

- 请你从输入 `X` 开始，推导 Multi-Head Attention 的完整计算流程。
- 每一步的张量 shape 是什么。
- 为什么要分多头。
- 为什么 attention score 要除以 `sqrt(d_k)`。
- 推理时怎么配合 KV cache。

### 标准公式

给定输入：
- `X`: `(B, T, D)`
- 头数 `H`
- 每个 head 的维度 `d = D / H`

线性映射：
- `Q = X W_Q`
- `K = X W_K`
- `V = X W_V`

reshape 后：
- `Q, K, V`: `(B, H, T, d)`

attention score：
- `S = Q K^T / sqrt(d)`
- `S`: `(B, H, T, T)`

加 mask 后 softmax：
- `A = softmax(S + mask)`

加权求和：
- `O = A V`
- `O`: `(B, H, T, d)`

拼接 heads：
- `O -> (B, T, D)`
- 再过输出投影 `W_O`

### 高频追问
- 为什么是多头而不是单头：不同子空间建模不同关系。
- 为什么除 `sqrt(d)`：防止点积随维度增大导致 softmax 饱和。
- 为什么复杂度高：attention matrix 是 `T x T`。
- 为什么 decode 时可以用 KV cache：历史 token 的 K/V 不变，不需要重复计算。

In [ ]:
print('scaled_dot_product_attention already defined above')

## 10. 手写 MHA 的简化实现

这类题常见要求：
- 不用现成 `nn.MultiheadAttention`
- 自己写 `W_q/W_k/W_v/W_o`
- 自己处理 reshape 和 transpose
- 支持 causal mask

面试里最容易错的点：
- `view` 前没有保证张量连续
- 头维和序列维 transpose 错了
- mask shape 不匹配
- 输出拼接后忘了过 `W_o`

In [ ]:
import torch
import torch.nn as nn


class SimpleMHA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        # x: (B, T, D) -> (B, H, T, d)
        B, T, D = x.shape
        x = x.view(B, T, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def merge_heads(self, x):
        # x: (B, H, T, d) -> (B, T, D)
        B, H, T, d = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, T, H * d)

    def forward(self, x, mask=None):
        q = self.split_heads(self.w_q(x))
        k = self.split_heads(self.w_k(x))
        v = self.split_heads(self.w_v(x))

        out, attn = scaled_dot_product_attention(q, k, v, mask)
        out = self.merge_heads(out)
        out = self.w_o(out)
        return out, attn


x = torch.randn(2, 8, 64)
mask = torch.tril(torch.ones(8, 8)).view(1, 1, 8, 8)
mha = SimpleMHA(d_model=64, num_heads=4)
y, attn = mha(x, mask)
print('y shape:', y.shape)
print('attn shape:', attn.shape)

## 11. MHA 常见面试题

### 题 1：为什么要除以 `sqrt(d_k)`
答题核心：
- 若 `q` 和 `k` 的元素均值为 0、方差为 1，则点积的方差会随着维度 `d_k` 增大。
- score 过大时，softmax 会过度尖锐，梯度变小，训练不稳定。
- 除以 `sqrt(d_k)` 相当于做尺度归一化。

### 题 2：为什么多头比单头好
答题核心：
- 不同头可以学习不同关系模式。
- 有的头关注局部语法，有的头关注长程依赖，有的头关注位置对齐。
- 多头本质上是在不同子空间做 attention。

### 题 3：MHA 的复杂度是多少
答题核心：
- 计算 score 的主要复杂度是 `O(B * H * T * T * d)`。
- 因为 `H * d = D`，通常写成 `O(B * T^2 * D)`。
- 真正的瓶颈通常不是线性层，而是 `T^2` 的注意力矩阵。

### 题 4：训练和推理时 MHA 有什么不同
答题核心：
- 训练时通常并行处理整段序列。
- 推理时是逐 token decode。
- 如果不用 KV cache，每次生成一个新 token 都要重算所有历史 K/V。
- 使用 KV cache 后，只需计算当前 token 的 Q、K、V，并复用历史 K/V。

## 12. KV Cache 也是高频题

面试常问：
- KV cache 缓存了什么
- 为什么只缓存 K/V，不缓存 Q
- 对显存和延迟有什么影响
- MHA、MQA、GQA 在 KV cache 上有什么区别

### 标准回答
- 对于历史 token，它们的 `K/V` 一旦计算出来就不会再变。
- 每次 decode 新 token，只需要算这个 token 的 `Q/K/V`。
- 然后用当前 `Q` 去和历史 `K` 做 attention，再对历史 `V` 加权。
- `Q` 不需要缓存，因为它只用于当前步。

### 高频追问：为什么 MQA/GQA 更省显存
- MHA 每个 head 都有一套 K/V。
- MQA 所有 query heads 共享一套 K/V。
- GQA 是若干 query heads 共享一组 K/V。
- 因此在长上下文推理时，MQA/GQA 能显著降低 KV cache 占用。

In [ ]:
def estimate_kv_cache_bytes(batch_size, seq_len, num_layers, num_kv_heads, head_dim, dtype_bytes=2):
    # K 和 V 各一份
    return batch_size * seq_len * num_layers * num_kv_heads * head_dim * 2 * dtype_bytes


batch_size = 1
seq_len = 8192
num_layers = 32
head_dim = 128

mha_bytes = estimate_kv_cache_bytes(batch_size, seq_len, num_layers, num_kv_heads=32, head_dim=head_dim)
gqa_bytes = estimate_kv_cache_bytes(batch_size, seq_len, num_layers, num_kv_heads=8, head_dim=head_dim)
mqa_bytes = estimate_kv_cache_bytes(batch_size, seq_len, num_layers, num_kv_heads=1, head_dim=head_dim)

print('MHA KV cache (MB):', mha_bytes / 1024 / 1024)
print('GQA KV cache (MB):', gqa_bytes / 1024 / 1024)
print('MQA KV cache (MB):', mqa_bytes / 1024 / 1024)

## 13. 位置编码会怎么考

### 高频题型
- 为什么 Transformer 需要位置编码
- 正弦位置编码公式是什么
- RoPE 为什么本质上是旋转
- RoPE 为什么对 attention 更友好
- 为什么 RoPE 在长文本场景里常见

### 一句话抓重点
- self-attention 本身对 token 顺序不敏感，所以必须显式注入位置信息。
- 绝对位置编码是“把位置加到 embedding 上”。
- RoPE 是“在 Q/K 空间里做位置相关旋转”，使相对位置信息自然进入点积。

### 面试时推荐回答路径
1. 先说没有位置编码就无法区分词序。
2. 再说绝对编码和相对编码差异。
3. 最后说 RoPE 通过二维旋转把相对位置信息融入 attention score。

In [ ]:
import torch


def rotate_half(x):
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    out = torch.stack((-x2, x1), dim=-1)
    return out.flatten(start_dim=-2)


def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin


T, d = 8, 16
x = torch.randn(2, 4, T, d)
theta = torch.arange(T).float().unsqueeze(1)
freq = torch.arange(d // 2).float().unsqueeze(0)
angles = theta * (1.0 / (10000 ** (2 * freq / d)))
cos = torch.repeat_interleave(torch.cos(angles), repeats=2, dim=-1).view(1, 1, T, d)
sin = torch.repeat_interleave(torch.sin(angles), repeats=2, dim=-1).view(1, 1, T, d)

y = apply_rope(x, cos, sin)
print('rope output shape:', y.shape)

## 14. 训练/推理优化题也常考

### FlashAttention
典型问题：
- FlashAttention 为什么更快
- 它优化的是算力还是访存

回答核心：
- 标准 attention 会显式构造大 `T x T` 矩阵，HBM 访存开销大。
- FlashAttention 用 tiling + online softmax，尽量在 SRAM/shared memory 里完成中间计算。
- 它减少了中间结果读写，因此 IO 更优。

### 并行训练
典型问题：
- 数据并行、张量并行、流水并行分别切什么
- 为什么单机多卡训练超大模型不够

回答核心：
- 数据并行：切 batch。
- 张量并行：切层内矩阵。
- 流水并行：切层。
- 实际大模型训练通常是 3D 并行组合。

## 15. 常见手写题清单

下面这些题出现频率很高，建议至少手写一遍：

- 手写 `softmax`
- 手写 `layer norm`
- 手写 `scaled dot-product attention`
- 手写 `multi-head attention`
- 手写 `causal mask`
- 手写 `RoPE`
- 手写 `top-k sampling`
- 手写 `top-p sampling`
- 手写 `cross entropy`
- 手写一个最小 Transformer block

如果岗位偏工程，还会考：
- 如何定位训练 loss 异常
- 如何排查梯度爆炸/消失
- 如何分析显存占用
- 如何提升 decode 吞吐
- 如何做 batch inference 和 continuous batching

## 16. 你应该重点准备的题

如果是大模型算法工程师，而不是纯 LeetCode 岗，优先级通常是：

### 第一梯队
- MHA 原理、shape、复杂度、mask
- KV cache
- RoPE
- Pre-LN/Post-LN
- FlashAttention 核心思想

### 第二梯队
- MQA/GQA
- 并行训练
- 推理优化
- 采样策略：greedy、beam search、top-k、top-p
- 多模态对齐训练的常见数据与损失设计

### 第三梯队
- MoE 路由
- 长上下文优化
- RLHF / DPO / GRPO 基础
- VLM 中视觉 token 压缩和 projector 设计

## 17. 高频面试问答模板

### 模板 1：请你介绍一下 MHA
建议回答：
1. 先说输入 `X` 经过线性层映射到 `Q/K/V`。
2. 再按 head 切分，在每个 head 上做 attention。
3. score 由 `QK^T / sqrt(d)` 得到，再加 mask、softmax。
4. 用 attention 权重对 `V` 加权求和。
5. 多个 head 拼接后再过输出投影。
6. 多头的作用是让模型在不同子空间建模不同关系。

### 模板 2：为什么推理要用 KV cache
建议回答：
1. 自回归生成是逐 token 解码。
2. 历史 token 的 K/V 不变，重复计算浪费算力。
3. KV cache 把历史 K/V 保存下来。
4. 新 token 到来时只计算当前步的 Q/K/V，并与历史 K/V 交互。
5. 这样延迟显著下降，但显存占用上升。

### 模板 3：为什么 FlashAttention 快
建议回答：
1. attention 的瓶颈常常是 IO，而不是纯算力。
2. 标准实现会产生巨大的 attention matrix 中间结果。
3. FlashAttention 通过分块和 online softmax，避免显式物化完整矩阵。
4. 因而减少 HBM 读写，提升速度并降低显存。

## 18. 自测题单

你可以尝试不用看资料，口头回答下面这些题：

1. MHA 的输入输出 shape 分别是什么。
2. 为什么 attention score 要缩放。
3. causal mask 和 padding mask 有什么区别。
4. KV cache 为什么只缓存 K/V。
5. MHA、MQA、GQA 的 KV cache 成本差异是什么。
6. RoPE 为什么能编码相对位置信息。
7. FlashAttention 优化的本质是什么。
8. prefill 和 decode 的计算模式有何不同。
9. 为什么长上下文推理更容易受 KV cache 限制。
10. 如果显存不够，你会优先优化哪些部分。

## 19. 最后建议

如果你时间有限，优先刷下面 6 个主题：
- MHA 手写与 shape 推导
- causal mask / padding mask
- KV cache
- RoPE
- FlashAttention
- MQA/GQA

这 6 个主题基本覆盖了大模型算法工程师面试中最容易被追问的主干问题。

如果你愿意，我下一步可以继续帮你补一个 notebook：
- 版本 A：`MHA/RoPE/KV cache` 重点手写题版
- 版本 B：偏面试问答版，适合直接背诵
- 版本 C：偏多模态/VLM 版，补视觉 token、cross-attention、Q-Former、projector